# Step 5 — Evaluate on Held-Out Segments

For every flight in `cleaned_data_final/`:
1. Load ADS-B before + after (the only inputs to reconstruction)
2. Reconstruct the gap with **Baseline**, **Kalman** and **BiGRU** — none of them see ADS-C
3. Reveal ADS-C as ground truth and measure error

Metrics computed per flight:
- **Geodesic position error** (km) — mean closest-point distance to ADS-C waypoints
- **Path-length error** (km) — difference in total route distance
- **Cross-track error** (km) — lateral deviation from the great-circle reference path

All flights are evaluated — no sampling.

## Cell 1 — Setup

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from pyproj import Geod
from scipy.interpolate import interp1d
from tqdm import tqdm

# ── Point to noel_part ────────────────────────────────────────────────────────
NOEL_DIR = Path(r"C:\Users\marko\Desktop\AeroML3\noel_part")
if not NOEL_DIR.exists():
    for _c in [Path("../noel_part"), Path("../../noel_part")]:
        if _c.resolve().exists(): NOEL_DIR = _c.resolve(); break

os.chdir(NOEL_DIR)
sys.path.insert(0, str(NOEL_DIR))

BASE_DIR  = Path(".")
CLEAN_DIR = BASE_DIR / "cleaned_data_final"
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

geod = Geod(ellps="WGS84")

from reconstruction import (TrajectoryBiGRU, FEATURE_COLS, TARGET_COLS,
                             SEQUENCE_LEN, reconstruct_gap_kalman,
                             reconstruct_gap, compute_features)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TrajectoryBiGRU(len(FEATURE_COLS), 128, 2, len(TARGET_COLS), 0.3).to(device)
model.load_state_dict(torch.load("models/BiGRU.pth", map_location=device))
model.eval()

flights = [f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir()
           for f in sorted(s.iterdir()) if f.is_dir()]

print(f"Working dir      : {os.getcwd()}")
print(f"Device           : {device}")
print(f"Total flights    : {len(flights)}")

## Cell 2 — Helper functions

In [ ]:
# ── Baseline: constant-speed great-circle arc ─────────────────────────────────
def reconstruct_forward_only(before_df, after_df, dt=15.0):
    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    n_steps = max(1, int(round((first_time - last_time).total_seconds() / dt)))
    lat0 = float(before_df["latitude"].iloc[-1]); lon0 = float(before_df["longitude"].iloc[-1])
    alt0 = float(before_df["altitude"].iloc[-1])
    lat1 = float(after_df["latitude"].iloc[0]);   lon1 = float(after_df["longitude"].iloc[0])
    alt1 = float(after_df["altitude"].iloc[0])
    pts  = geod.npts(lon0, lat0, lon1, lat1, n_steps)
    lats = np.array([p[1] for p in pts]); lons = np.array([p[0] for p in pts])
    alts = np.linspace(alt0, alt1, n_steps)
    timestamps = [last_time + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    result = pd.DataFrame({"latitude": lats, "longitude": lons, "altitude": alts})
    result["timestamp"] = timestamps; result["interpolated"] = True
    return result

# ── Retime: keep shape, fix speed ─────────────────────────────────────────────
def retime_to_constant_speed(recon_df, before_df, after_df, dt=15.0):
    lats = recon_df["latitude"].values; lons = recon_df["longitude"].values
    alts = recon_df["altitude"].values if "altitude" in recon_df.columns else np.zeros(len(recon_df))
    cum_dist = np.zeros(len(lats))
    for i in range(1, len(lats)):
        _, _, d = geod.inv(float(lons[i-1]), float(lats[i-1]), float(lons[i]), float(lats[i]))
        cum_dist[i] = cum_dist[i-1] + abs(d)
    total_dist = cum_dist[-1]
    if total_dist == 0: return recon_df
    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    total_sec  = (first_time - last_time).total_seconds()
    n_steps    = max(1, int(round(total_sec / dt)))
    time_fracs = np.clip([dt*(i+1)/total_sec for i in range(n_steps)], 0, 1)
    old_fracs  = cum_dist / total_dist
    f_la  = interp1d(old_fracs, lats, bounds_error=False, fill_value=(lats[0], lats[-1]))
    f_lo  = interp1d(old_fracs, lons, bounds_error=False, fill_value=(lons[0], lons[-1]))
    f_alt = interp1d(old_fracs, alts, bounds_error=False, fill_value=(alts[0], alts[-1]))
    new_ts = [last_time + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    return pd.DataFrame({"latitude": f_la(time_fracs), "longitude": f_lo(time_fracs),
                         "altitude": f_alt(time_fracs), "timestamp": new_ts, "interpolated": True})

# ── Geodesic position error (mean closest-point distance) ─────────────────────
def geodesic_error_km(recon_df, truth_df):
    if len(truth_df) == 0 or len(recon_df) == 0: return np.nan
    def hav(la1, lo1, la2, lo2):
        la1,lo1,la2,lo2 = map(np.radians,[la1,lo1,la2,lo2])
        dlat,dlon = la2-la1, lo2-lo1
        a = np.sin(dlat/2)**2 + np.cos(la1)*np.cos(la2)*np.sin(dlon/2)**2
        return 2*6_371_000*np.arcsin(np.sqrt(np.clip(a,0,1)))
    errs = []
    for _, row in truth_df.iterrows():
        dists = hav(row["latitude"], row["longitude"],
                    recon_df["latitude"].values, recon_df["longitude"].values)
        errs.append(dists.min())
    return float(np.mean(errs)) / 1000

# ── Path-length error ─────────────────────────────────────────────────────────
def path_length_km(df):
    lats = df["latitude"].values; lons = df["longitude"].values
    if len(lats) < 2: return 0.0
    _, _, dists = geod.inv(lons[:-1], lats[:-1], lons[1:], lats[1:])
    return float(np.nansum(np.abs(dists))) / 1000

def path_length_error_km(recon_df, truth_df):
    return abs(path_length_km(recon_df) - path_length_km(truth_df))

# ── Cross-track error ─────────────────────────────────────────────────────────
def cross_track_error_km(recon_df, start_lat, start_lon, end_lat, end_lon):
    """Mean perpendicular distance from the great-circle reference path."""
    if len(recon_df) == 0: return np.nan
    errors = []
    bearing_ref = np.radians(
        (np.degrees(np.arctan2(
            np.sin(np.radians(end_lon - start_lon)) * np.cos(np.radians(end_lat)),
            np.cos(np.radians(start_lat)) * np.sin(np.radians(end_lat)) -
            np.sin(np.radians(start_lat)) * np.cos(np.radians(end_lat)) *
            np.cos(np.radians(end_lon - start_lon))
        )) + 360) % 360
    )
    R = 6_371_000.0
    for _, row in recon_df.iterrows():
        lat1, lon1 = np.radians(start_lat), np.radians(start_lon)
        lat2, lon2 = np.radians(row["latitude"]), np.radians(row["longitude"])
        dlat = lat2 - lat1; dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        d13 = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
        bearing_13 = np.arctan2(
            np.sin(dlon)*np.cos(lat2),
            np.cos(lat1)*np.sin(lat2) - np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
        )
        xte = np.arcsin(np.clip(np.sin(d13) * np.sin(bearing_13 - bearing_ref), -1, 1)) * R
        errors.append(abs(xte))
    return float(np.mean(errors)) / 1000

print("Helper functions defined.")

## Cell 3 — Evaluate ALL flights

This cell runs baseline, Kalman and BiGRU on every flight and measures all three error metrics.
Expected runtime: ~10-30 minutes depending on number of flights and hardware.

In [ ]:
records = []
skipped = 0

print(f"Evaluating {len(flights)} flights ...")
print("(This may take 10-30 minutes)\n")

for flight_dir in tqdm(flights, desc="Evaluating"):
    try:
        # ── Load data ─────────────────────────────────────────────────────────
        _ap  = next((flight_dir/p for p in ["adsc_clean.parquet","adsc.parquet"]
                     if (flight_dir/p).exists()), None)
        _bp  = next((flight_dir/p for p in ["adsb_before_clean.parquet","adsb_before.parquet"]
                     if (flight_dir/p).exists()), None)
        _afp = next((flight_dir/p for p in ["adsb_after_clean.parquet","adsb_after.parquet"]
                     if (flight_dir/p).exists()), None)

        if not (_ap and _bp and _afp):
            skipped += 1; continue

        adsc = pd.read_parquet(str(_ap)).dropna(subset=["latitude","longitude"])
        bef  = pd.read_parquet(str(_bp)).dropna(subset=["latitude","longitude"])
        aft  = pd.read_parquet(str(_afp)).dropna(subset=["latitude","longitude"])

        for df in [adsc, bef, aft]:
            df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_localize(None)
            df.sort_values("timestamp", inplace=True)
            df.reset_index(drop=True, inplace=True)

        if len(bef) < 4 or len(aft) < 4 or len(adsc) < 2:
            skipped += 1; continue

        # ── Gap boundaries ────────────────────────────────────────────────────
        t_gap_start = bef["timestamp"].iloc[-1]
        t_gap_end   = aft["timestamp"].iloc[0]
        gap_min = (t_gap_end - t_gap_start).total_seconds() / 60

        if gap_min < 5:
            skipped += 1; continue

        # ADS-C strictly inside the gap (held-out ground truth)
        adsc_gap = adsc[
            (adsc["timestamp"] > t_gap_start) &
            (adsc["timestamp"] < t_gap_end)
        ].reset_index(drop=True)

        if len(adsc_gap) == 0:
            skipped += 1; continue

        # Trim ADS-B to 30 min around gap boundary
        bef_trim = bef[bef["timestamp"] >= t_gap_start - pd.Timedelta(minutes=30)].reset_index(drop=True)
        aft_trim = aft[aft["timestamp"] <= t_gap_end   + pd.Timedelta(minutes=30)].reset_index(drop=True)

        if len(bef_trim) < 2 or len(aft_trim) < 2:
            skipped += 1; continue

        # Gap endpoints for cross-track reference
        start_lat = float(bef_trim["latitude"].iloc[-1])
        start_lon = float(bef_trim["longitude"].iloc[-1])
        end_lat   = float(aft_trim["latitude"].iloc[0])
        end_lon   = float(aft_trim["longitude"].iloc[0])

        # ── Reconstruct — none of the methods see ADS-C ───────────────────────
        recon_base = reconstruct_forward_only(bef_trim, aft_trim)

        recon_kalman_raw = compute_features(reconstruct_gap_kalman(bef_trim, aft_trim))
        recon_kalman     = retime_to_constant_speed(recon_kalman_raw, bef_trim, aft_trim)

        recon_bigru_raw = compute_features(
            reconstruct_gap(model, bef_trim, aft_trim, FEATURE_COLS, TARGET_COLS, SEQUENCE_LEN, device)
        )
        recon_bigru = retime_to_constant_speed(recon_bigru_raw, bef_trim, aft_trim)

        # ── Measure errors against ADS-C ground truth ─────────────────────────
        records.append({
            "flight":               flight_dir.name,
            "step":                 flight_dir.parent.name,
            "gap_minutes":          round(gap_min, 1),
            "adsc_waypoints":       len(adsc_gap),

            # Geodesic position error (km)
            "baseline_geo_km":      geodesic_error_km(recon_base,   adsc_gap),
            "kalman_geo_km":        geodesic_error_km(recon_kalman, adsc_gap),
            "bigru_geo_km":         geodesic_error_km(recon_bigru,  adsc_gap),

            # Path-length error (km)
            "baseline_path_err_km": path_length_error_km(recon_base,   adsc_gap),
            "kalman_path_err_km":   path_length_error_km(recon_kalman, adsc_gap),
            "bigru_path_err_km":    path_length_error_km(recon_bigru,  adsc_gap),

            # Cross-track error (km)
            "baseline_xte_km":      cross_track_error_km(recon_base,   start_lat, start_lon, end_lat, end_lon),
            "kalman_xte_km":        cross_track_error_km(recon_kalman, start_lat, start_lon, end_lat, end_lon),
            "bigru_xte_km":         cross_track_error_km(recon_bigru,  start_lat, start_lon, end_lat, end_lon),
        })

    except Exception as _e:
        skipped += 1
        continue

eval_df = pd.DataFrame(records)
eval_df.to_csv(OUT_DIR / "evaluation_results.csv", index=False)

print(f"\nFlights evaluated : {len(eval_df)}")
print(f"Flights skipped   : {skipped}")
print(f"Saved → {OUT_DIR / 'evaluation_results.csv'}")

## Cell 4 — Results summary

In [ ]:
print("=" * 60)
print("EVALUATION RESULTS — All flights")
print("=" * 60)

print(f"\nFlights evaluated : {len(eval_df)}")
print(f"Avg gap duration  : {eval_df['gap_minutes'].mean():.1f} min")
print(f"Avg ADS-C waypts  : {eval_df['adsc_waypoints'].mean():.1f}")

# ── Geodesic position error ───────────────────────────────────────────────────
print("\n── Geodesic Position Error (mean closest-point distance to ADS-C) ──")
print(f"{'Method':<12}  {'Mean (km)':>10}  {'Median (km)':>12}  {'P90 (km)':>10}  {'Improvement':>12}")
print("-" * 62)
base_mean = eval_df["baseline_geo_km"].mean()
for method, col in [("Baseline","baseline_geo_km"), ("Kalman","kalman_geo_km"), ("BiGRU","bigru_geo_km")]:
    vals = eval_df[col].dropna()
    imp  = (1 - vals.mean()/base_mean)*100 if method != "Baseline" else 0.0
    imp_str = "—" if method == "Baseline" else f"{imp:+.1f}%"
    print(f"  {method:<10}  {vals.mean():>10.1f}  {vals.median():>12.1f}  "
          f"{vals.quantile(0.9):>10.1f}  {imp_str:>12}")

# ── Path-length error ─────────────────────────────────────────────────────────
print("\n── Path-Length Error (|reconstructed length − ADS-C length|) ──")
print(f"{'Method':<12}  {'Mean (km)':>10}  {'Median (km)':>12}")
print("-" * 38)
for method, col in [("Baseline","baseline_path_err_km"), ("Kalman","kalman_path_err_km"), ("BiGRU","bigru_path_err_km")]:
    vals = eval_df[col].dropna()
    print(f"  {method:<10}  {vals.mean():>10.1f}  {vals.median():>12.1f}")

# ── Cross-track error ─────────────────────────────────────────────────────────
print("\n── Cross-Track Error (lateral deviation from great-circle path) ──")
print(f"{'Method':<12}  {'Mean (km)':>10}  {'Median (km)':>12}")
print("-" * 38)
for method, col in [("Baseline","baseline_xte_km"), ("Kalman","kalman_xte_km"), ("BiGRU","bigru_xte_km")]:
    vals = eval_df[col].dropna()
    print(f"  {method:<10}  {vals.mean():>10.1f}  {vals.median():>12.1f}")

# ── Win rates ─────────────────────────────────────────────────────────────────
print("\n── Method Win Rates (which method has lowest geodesic error per flight) ──")
wins = {
    "Baseline": (eval_df["baseline_geo_km"] <= eval_df[["kalman_geo_km","bigru_geo_km"]].min(axis=1)).sum(),
    "Kalman"  : (eval_df["kalman_geo_km"]   <= eval_df[["baseline_geo_km","bigru_geo_km"]].min(axis=1)).sum(),
    "BiGRU"   : (eval_df["bigru_geo_km"]    <= eval_df[["baseline_geo_km","kalman_geo_km"]].min(axis=1)).sum(),
}
for method, w in wins.items():
    print(f"  {method:<10}: {w:>4} flights  ({w/len(eval_df)*100:.1f}%)")

## Cell 5 — Error distribution plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

methods = [
    ("Baseline", "baseline_geo_km",      "#F44336"),
    ("Kalman",   "kalman_geo_km",         "#00BCD4"),
    ("BiGRU",    "bigru_geo_km",          "#4CAF50"),
]

# ── Geodesic error distributions ──────────────────────────────────────────────
ax = axes[0]
data = [eval_df[col].dropna() for _, col, _ in methods]
bp = ax.boxplot(data, patch_artist=True, widths=0.5)
for patch, (_, _, color) in zip(bp["boxes"], methods):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticklabels([m for m,_,_ in methods])
ax.set_ylabel("Mean geodesic error (km)")
ax.set_title("Geodesic Position Error Distribution")
ax.grid(axis="y", alpha=0.3)
for i, (m, col, _) in enumerate(methods):
    ax.text(i+1, eval_df[col].mean(), f"{eval_df[col].mean():.1f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

# ── Path-length error ─────────────────────────────────────────────────────────
ax = axes[1]
path_methods = [
    ("Baseline", "baseline_path_err_km", "#F44336"),
    ("Kalman",   "kalman_path_err_km",   "#00BCD4"),
    ("BiGRU",    "bigru_path_err_km",    "#4CAF50"),
]
data2 = [eval_df[col].dropna() for _, col, _ in path_methods]
bp2 = ax.boxplot(data2, patch_artist=True, widths=0.5)
for patch, (_, _, color) in zip(bp2["boxes"], path_methods):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticklabels([m for m,_,_ in path_methods])
ax.set_ylabel("Path-length error (km)")
ax.set_title("Path-Length Error Distribution")
ax.grid(axis="y", alpha=0.3)

# ── Cross-track error ─────────────────────────────────────────────────────────
ax = axes[2]
xte_methods = [
    ("Baseline", "baseline_xte_km", "#F44336"),
    ("Kalman",   "kalman_xte_km",   "#00BCD4"),
    ("BiGRU",    "bigru_xte_km",    "#4CAF50"),
]
data3 = [eval_df[col].dropna() for _, col, _ in xte_methods]
bp3 = ax.boxplot(data3, patch_artist=True, widths=0.5)
for patch, (_, _, color) in zip(bp3["boxes"], xte_methods):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticklabels([m for m,_,_ in xte_methods])
ax.set_ylabel("Cross-track error (km)")
ax.set_title("Cross-Track Error Distribution")
ax.grid(axis="y", alpha=0.3)

plt.suptitle(f"Reconstruction Evaluation — {len(eval_df)} Flights", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "evaluation_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {OUT_DIR / 'evaluation_distributions.png'}")

## Cell 6 — Error vs gap duration

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for method, col, color in methods:
    ax.scatter(eval_df["gap_minutes"], eval_df[col],
               alpha=0.4, s=20, color=color, label=method)

# Trend lines
for method, col, color in methods:
    _x = eval_df["gap_minutes"].values
    _y = eval_df[col].values
    mask = np.isfinite(_x) & np.isfinite(_y)
    z = np.polyfit(_x[mask], _y[mask], 1)
    p = np.poly1d(z)
    _xr = np.linspace(_x[mask].min(), _x[mask].max(), 100)
    ax.plot(_xr, p(_xr), color=color, linewidth=2, linestyle="--")

ax.set_xlabel("Gap duration (minutes)")
ax.set_ylabel("Geodesic error (km)")
ax.set_title("Reconstruction Error vs Gap Duration")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "error_vs_gap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {OUT_DIR / 'error_vs_gap.png'}")

## Cell 7 — Per-flight breakdown table

In [ ]:
# Show worst and best flights per method
print("Top 10 flights where BiGRU beats baseline most:")
eval_df["bigru_improvement_km"] = eval_df["baseline_geo_km"] - eval_df["bigru_geo_km"]
top = eval_df.nlargest(10, "bigru_improvement_km")[
    ["flight","gap_minutes","baseline_geo_km","kalman_geo_km","bigru_geo_km","bigru_improvement_km"]
].round(1)
display(top)

print("\nTop 10 flights where baseline beats both Kalman and BiGRU:")
both_worse = eval_df[
    (eval_df["baseline_geo_km"] < eval_df["kalman_geo_km"]) &
    (eval_df["baseline_geo_km"] < eval_df["bigru_geo_km"])
].nsmallest(10, "baseline_geo_km")[
    ["flight","gap_minutes","baseline_geo_km","kalman_geo_km","bigru_geo_km"]
].round(1)
display(both_worse)

# Save full per-flight table
eval_df.to_csv(OUT_DIR / "evaluation_results.csv", index=False)
print(f"\nFull results saved → {OUT_DIR / 'evaluation_results.csv'}")
print(f"Total flights: {len(eval_df)}")